In [ ]:
# Setup
%pip -q install gdown
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt


In [ ]:
# Setup
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}


In [ ]:
# Shared: data
def load_data(path):
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]


In [ ]:
# Shared: model
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model


In [ ]:
# (a): PGD attacks
def clip_box(x, delta):
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.empty_like(x).uniform_(-eps, eps) if rand_init else torch.zeros_like(x)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = (delta + alpha * grad.sign()).clamp(-eps, eps)
        delta = clip_box(x, delta)
    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.randn_like(x)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / l2_norm(delta).clamp_min(1e-12) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad / l2_norm(grad).clamp_min(1e-12)
        delta = delta * (eps / l2_norm(delta).clamp_min(1e-12)).clamp(max=1)
        delta = clip_box(x, delta)
    return (x + delta).detach()


In [ ]:
# Shared: attacks
def project_l1(delta, eps):
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)


In [ ]:
# Shared: evaluation
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc


In [ ]:
# Shared: training
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    model.epoch_times = []
    for epoch in range(epochs):
        epoch_start = time.perf_counter()
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        model.epoch_times.append(time.perf_counter() - epoch_start)
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model


In [ ]:
def single_norm_loss(model, x, y, norm):
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(x_adv), y)


In [ ]:
import json
import math
import pandas as pd
from IPython.display import display

results = {"base": test_model(load_base_model()), "ramp": {}, "plan1": {}}
single_results = {}
for norm in NORMS:
    model = adv_train(load_base_model(), single_norm_loss, norm=norm, seed=0)
    single_results[norm] = {"accuracy": test_model(model), "epoch_seconds": model.epoch_times}
    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, f"single_{norm}.pt")
    with open("problem3_single_results.json", "w") as f:
        json.dump(single_results, f, indent=2)
    del model


In [ ]:
single_table = {"Base": results["base"]}
for norm in NORMS:
    single_table[norm] = single_results[norm]["accuracy"]
single_table = pd.DataFrame(single_table).T
display(single_table)
display(single_table[list(NORMS)].idxmin(axis=1))
display(single_table - single_table.loc["Base"])


In [ ]:
def ramp_loss(model, x, y, lam=0.5):
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()
    z_q = model(x_q)
    z_r = model(x_r)
    loss_q = F.cross_entropy(z_q, y, reduction="none")
    loss_r = F.cross_entropy(z_r, y, reduction="none")
    correct = z_q.argmax(1) == y
    kl = z_q.sum() * 0
    if correct.any():
        kl = F.kl_div(F.log_softmax(z_r[correct], dim=1),
                      F.softmax(z_q[correct], dim=1), reduction="batchmean")
    return torch.maximum(loss_q, loss_r).mean() + lam * kl


def gradient_projection(g_n, g_a):
    cosine = F.cosine_similarity(g_n.flatten(), g_a.flatten(), dim=0, eps=1e-12)
    return cosine.clamp_min(0) * g_n


In [ ]:
for seed in (0, 1):
    model = adv_train(load_base_model(), ramp_loss, gp_beta=0.5, seed=seed)
    results["ramp"][seed] = {"accuracy": test_model(model), "epoch_seconds": model.epoch_times}
    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, f"ramp_seed{seed}.pt")
    with open("problem3c_results.json", "w") as f:
        json.dump(results, f, indent=2)
    del model


In [ ]:
rows = [{"Method": "Base", **results["base"], "s/epoch": float("nan")}]
for name, r in list(single_results.items()) + [("RAMP", results["ramp"][0])]:
    rows.append({"Method": name, **r["accuracy"], "s/epoch": np.mean(r["epoch_seconds"])})
display(pd.DataFrame(rows).set_index("Method").round(2))


In [ ]:
teacher = None
repair_step = 0


def my_loss(model, x, y):
    global repair_step
    norm = "l1" if repair_step % 2 == 0 else "l2"
    repair_step += 1
    model.eval()
    x_repair = attack(model, x, y, norm)
    x_keep = attack(model, x, y, "linf")
    with torch.no_grad():
        target = teacher(x_keep).softmax(1)
        current = model(x_keep).softmax(1)
        gate = (target.argmax(1) == y) & (current.gather(1, y[:, None]).squeeze(1) < target.gather(1, y[:, None]).squeeze(1))
    model.train()
    repair = F.cross_entropy(model(x_repair), y)
    kl = F.kl_div(F.log_softmax(model(x_keep), dim=1), target, reduction="none").sum(1)
    preserve = (kl * gate).sum() / gate.sum().clamp_min(1)
    return repair + preserve


In [ ]:
for seed in (0, 1):
    teacher = load_base_model().eval().requires_grad_(False)
    repair_step = 0
    model = adv_train(load_base_model(), my_loss, seed=seed)
    results["plan1"][seed] = {"accuracy": test_model(model), "epoch_seconds": model.epoch_times}
    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, f"plan1_seed{seed}.pt")
    with open("problem3c_results.json", "w") as f:
        json.dump(results, f, indent=2)
    del model
    teacher = None


In [ ]:
mean_plan1 = np.mean([sum(r["epoch_seconds"]) for r in results["plan1"].values()])
mean_ramp = np.mean([sum(r["epoch_seconds"]) for r in results["ramp"].values()])
if mean_plan1 > mean_ramp:
    results["ramp_matched"] = {}
    for seed in (0, 1):
        epochs = math.ceil(10 * mean_plan1 / mean_ramp)
        while True:
            model = adv_train(load_base_model(), ramp_loss, epochs=epochs, gp_beta=0.5, seed=seed)
            if sum(model.epoch_times) >= mean_plan1:
                break
            del model
            epochs += 1
        results["ramp_matched"][seed] = {"accuracy": test_model(model), "epoch_seconds": model.epoch_times}
        torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, f"ramp_matched_seed{seed}.pt")
        del model
    with open("problem3c_results.json", "w") as f:
        json.dump(results, f, indent=2)


In [ ]:
rows = [{"Method": "Base", **results["base"], "s/epoch": float("nan"), "Total seconds": float("nan")}]
for method in ("ramp", "plan1", "ramp_matched"):
    if method in results:
        runs = list(results[method].values())
        row = {"Method": method}
        row.update({key: np.mean([r["accuracy"][key] for r in runs]) for key in results["base"]})
        row["s/epoch"] = np.mean([np.mean(r["epoch_seconds"]) for r in runs])
        row["Total seconds"] = np.mean([sum(r["epoch_seconds"]) for r in runs])
        rows.append(row)
display(pd.DataFrame(rows).set_index("Method").round(2))
times = {f"{method}, seed {seed}": r["epoch_seconds"] for method in results if method != "base" for seed, r in results[method].items()}
display(pd.DataFrame({name: pd.Series(values, index=range(1, len(values) + 1)) for name, values in times.items()}).rename_axis("Epoch").round(1))


In [ ]:
def ablation_loss(model, x, y):
    global repair_step
    norm = "l1" if repair_step % 2 == 0 else "l2"
    repair_step += 1
    model.eval()
    x_repair = attack(model, x, y, norm)
    x_keep = attack(model, x, y, "linf")
    with torch.no_grad():
        target = teacher(x_keep).softmax(1)
        current = model(x_keep).softmax(1)
        gate = (target.argmax(1) == y) & (current.gather(1, y[:, None]).squeeze(1) < target.gather(1, y[:, None]).squeeze(1))
    model.train()
    repair = F.cross_entropy(model(x_repair), y)
    kl = F.kl_div(F.log_softmax(model(x_keep), dim=1), target, reduction="none").sum(1)
    preserve = (kl * gate).sum() / gate.sum().clamp_min(1)
    return repair + 0 * preserve


In [ ]:
# ablation training
ablation_results = {}
for seed in (0, 1):
    teacher = load_base_model().eval().requires_grad_(False)
    repair_step = 0
    model = adv_train(load_base_model(), ablation_loss, seed=seed)
    ablation_results[seed] = {"accuracy": test_model(model), "epoch_seconds": model.epoch_times}
    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, f"ablation_seed{seed}.pt")
    with open("problem3c_ablation_results.json", "w") as f:
        json.dump(ablation_results, f, indent=2)
    del model
    teacher = None


In [ ]:
full_mean = {}
ablation_mean = {}
for key in results["base"]:
    full_mean[key] = (results["plan1"][0]["accuracy"][key] + results["plan1"][1]["accuracy"][key]) / 2
    ablation_mean[key] = (ablation_results[0]["accuracy"][key] + ablation_results[1]["accuracy"][key]) / 2
ablation_comparison = pd.DataFrame([full_mean, ablation_mean], index=["Full method", "Preservation weight 0"])
ablation_comparison.loc["Ablation minus full (pp)"] = ablation_comparison.iloc[1] - ablation_comparison.iloc[0]
display(ablation_comparison.round(2))


In [ ]:
ramp_mean = {}
for key in results["base"]:
    ramp_mean[key] = (results["ramp"][0]["accuracy"][key] + results["ramp"][1]["accuracy"][key]) / 2
means = pd.DataFrame([results["base"], ramp_mean, full_mean, ablation_mean], index=["Base", "RAMP", "Proposed", "Ablation"])
display(means.round(2))
for name in ["Base", "RAMP", "Ablation"]:
    print("Proposed minus", name, "(pp)")
    display((means.loc["Proposed"] - means.loc[name]).round(2))


In [ ]:
strong_results = {}
for method in ("ramp", "plan1"):
    seed = 0
    if results[method][1]["accuracy"]["union"] > results[method][0]["accuracy"]["union"]:
        seed = 1
    model = PreActResNet18()
    model.load_state_dict(torch.load(f"{method}_seed{seed}.pt", map_location="cpu"))
    model = model.to(device)
    strong_results[method] = {"seed": seed, "default": results[method][seed]["accuracy"], "strong": test_model(model, steps=200, restarts=5)}
    with open("problem3c_strong_results.json", "w") as f:
        json.dump(strong_results, f, indent=2)
    del model
rows = []
for method, r in strong_results.items():
    for setting in ("default", "strong"):
        rows.append({"Method": method, "Seed": r["seed"], "Evaluation": setting, **r[setting]})
display(pd.DataFrame(rows))


In [ ]:
drops = {}
for method, r in strong_results.items():
    drops[method] = {}
    for key in r["default"]:
        drops[method][key] = r["default"][key] - r["strong"][key]
display(pd.DataFrame(drops).T.rename_axis("Default minus strong (pp)").round(2))
